In [0]:
%sql
create catalog if not exists medallion;
use catalog medallion;
create volume if not exists landing;
create schema if not exists bronze;
create schema if not exists silver;
create schema if not exists gold;

In [0]:
from pyspark.sql import functions as F

# Caminho da landing
base_path = "/Volumes/medallion/default/landing/"

# Função para ler o CSV e transformar em tabela Delta
def create_bronze_table(file_name, table_name):
    df = (
        spark.read
        # primeira linha considerada como nome da coluna
        .option("header", True)
        # o Databricks infere o tipo de cada coluna com base nos dados
        .option("inferSchema", True)
        .csv(base_path + file_name)
    )

    df = df.withColumn(
        # adição de coluna para saber quando foi a ingestão daquele dado
        "timestamp_ingestion",
        F.current_timestamp()
    )

    # Escrita da tabela Bronze
    df.write \
      .option("mergeSchema", "true") \
      .mode("overwrite") \
      .saveAsTable(table_name)

In [0]:
# confirma criação do schema bronze e chama a função
spark.sql(f"create schema if not exists bronze")
create_bronze_table("olist_customers_dataset.csv", "medallion.bronze.tb_customers")
create_bronze_table("olist_geolocation_dataset.csv", "medallion.bronze.tb_geolocalizacao")
create_bronze_table("olist_order_items_dataset.csv", "medallion.bronze.tb_order_items")
create_bronze_table("olist_order_payments_dataset.csv", "medallion.bronze.tb_order_payments")
create_bronze_table("olist_order_reviews_dataset.csv", "medallion.bronze.tb_order_reviews")
create_bronze_table("olist_orders_dataset.csv", "medallion.bronze.tb_orders")
create_bronze_table("olist_products_dataset.csv", "medallion.bronze.tb_products")
create_bronze_table("olist_sellers_dataset.csv", "medallion.bronze.tb_sellers")
create_bronze_table("product_category_name_translation.csv", "medallion.bronze.tb_product_category_name_translation")

API BCB puxar cotação Dólar

In [0]:
import requests
from pyspark.sql.functions import current_timestamp

# pegando a data mais recente e mais antiga da tabela de pedidos para usar de parametro na requisição a API
df_datas = spark.sql("""
SELECT 
    date_format(max(order_purchase_timestamp), 'MM-dd-yyyy') as max_date, 
    date_format(min(order_purchase_timestamp), 'MM-dd-yyyy') as min_date
FROM medallion.bronze.tb_orders
WHERE CAST(order_purchase_timestamp AS STRING) LIKE '20%'
""")

row = df_datas.collect()[0]
data_inicio = row["min_date"]
data_fim = row["max_date"]

url = (f"https://olinda.bcb.gov.br/olinda/servico/PTAX/versao/v1/odata/"
       f"CotacaoDolarPeriodo(dataInicial=@dataInicial,dataFinalCotacao=@dataFinalCotacao)?"
       f"@dataInicial='{data_inicio}'&@dataFinalCotacao='{data_fim}'"
       f"&$select=dataHoraCotacao,cotacaoCompra&$format=json")

# Fazendo a requisição
response = requests.get(url)

# Só faz alterações nas tabelas se a requisição for bem sucedida
if response.status_code == 200:
    # Guardando o valor da resposta
    data = response.json()['value']
    
    # Criando o DataFrame base para a tabela e adicionando as colunas fornecidas pela API 
    df_dolar = spark.createDataFrame(data)
    df_dolar = df_dolar.withColumn("timestamp_ingestion", current_timestamp())
    
    # Escrevendo os dados na tabela de cotacao_dolar
    (df_dolar.write
     .mode("overwrite")
     .option("overwriteSchema", "true")
     .saveAsTable("medallion.bronze.tb_cotacao_dolar"))
    
else:
    print(f"❌ Erro ao acessar API: {response.status_code}")